In [1]:
import os
import cv2
import shutil
import zipfile
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# --- [경로 설정] ---
LOCAL_EXTRACT_DIR = "/content/temp_images"
SAVE_DIR_DRIVE = "/content/drive/MyDrive/Box_Label" # 결과(이미지+라벨) 저장 경로
WEIGHTS_PATH = "/content/drive/MyDrive/vm/best.pt" # 👈 본인의 best.pt 경로로 수정!

os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR_DRIVE, exist_ok=True)

# 2. 필수 라이브러리 설치 (ultralytics)
try:
    from ultralytics import YOLO
    print("✅ YOLO 라이브러리 준비 완료")
except ImportError:
    !pip install -q ultralytics
    from ultralytics import YOLO

# 3. ZIP 파일 압축 해제
ZIP_PATH = "/content/drive/MyDrive/dataset/images.zip"
if not os.path.exists(LOCAL_EXTRACT_DIR) or len(os.listdir(LOCAL_EXTRACT_DIR)) == 0:
    print("📂 로컬 디스크로 압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)

# 4. 모델 로드 (본인의 best.pt)
model = YOLO(WEIGHTS_PATH)

# 5. 이미지 목록 가져오기
image_files = [os.path.join(r, f) for r, d, fs in os.walk(LOCAL_EXTRACT_DIR)
               for f in fs if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
image_files = sorted(image_files)
total_images = len(image_files)

# 6. 추론 및 라벨 생성 루프
saved_count = 0
skipped_count = 0

print(f"🚀 작업 시작: 총 {total_images}장 대상")

for index, img_path in enumerate(image_files):
    filename = os.path.basename(img_path)
    label_filename = os.path.splitext(filename)[0] + ".txt"
    label_path = os.path.join(SAVE_DIR_DRIVE, label_filename)

    # [이어하기] 이미 라벨 파일이 있다면 스킵
    if os.path.exists(label_path):
        skipped_count += 1
        continue

    # 모델 추론 (conf 설정 가능)
    results = model(img_path, conf=0.25, verbose=False)[0]

    # YOLO 포맷 좌표 추출
    # results.boxes.xywhn은 [class_id, x_center, y_center, width, height] 정규화된 값을 가짐
    boxes = results.boxes
    if len(boxes) > 0:
        label_lines = []
        for box in boxes:
            # 클래스 번호, 정규화된 좌표(xywhn) 가져오기
            cls = int(box.cls[0])
            xywhn = box.xywhn[0].tolist() # [x_c, y_c, w, h]

            line = f"{cls} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}\n"
            label_lines.append(line)

        # 파일 저장 (라벨 .txt)
        with open(label_path, "w") as f:
            f.writelines(label_lines)

        # 원본 이미지도 결과 폴더로 복사 (학습 데이터셋 형태 유지)
        shutil.copy(img_path, os.path.join(SAVE_DIR_DRIVE, filename))
        saved_count += 1

    if (index + 1) % 100 == 0 or (index + 1) == total_images:
        print(f"📊 [{index + 1}/{total_images}] 완료 (저장: {saved_count}, 스킵: {skipped_count})")

print("-" * 30)
print(f"🏁 작업 완료! 결과물은 '{SAVE_DIR_DRIVE}'에서 확인하세요.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📂 로컬 디스크로 압축 해제 중...
🚀 작업 시작: 총 500장 대상
📊 [100/500] 완료 (저장: 100, 스킵: 0)
📊 [200/500] 완료 (저장: 200, 스킵: 0)
📊 [300/500] 완료 (저장: 300, 스킵: 0)
📊 [400/500] 완료 (저장: 400, 스킵: 0)
📊 [500/500] 완료 (저장: 500, 스킵: 0)
------------------------------
🏁 작업 완료! 결과물은 '/content/drive/MyDrive/Box_Label'에서 확인하세요.


In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

# 드라이브 내 패키지 저장 폴더 생성
nb_path = '/content/drive/MyDrive/colab_packages'
os.makedirs(nb_path, exist_ok=True)

# 해당 폴더에 설치 (처음 한 번만 실행)
!pip install --target={nb_path} inference-gpu roboflow